# Statistical Baseline Experiments on TSB-AD-U

This notebook evaluates configurable statistical methods on the univariate TSB-AD-U dataset.

How to use this notebook:
1. Edit only the **Configuration** cell.
2. Run all remaining cells from top to bottom.
3. Inspect the summary tables.
4. Export results to CSV.

The default setup runs a quick smoke test on 10 files using:
- `Sub_PCA`
- `KShapeAD`
- `Series2Graph`
- `KMeansAD`

In [16]:
from __future__ import annotations

import random
import re
import time
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import TSB_AD.model_wrapper as model_wrapper
from IPython.display import display
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.preprocessing import MinMaxScaler

from TSB_AD.evaluation.metrics import get_metrics
from TSB_AD.model_wrapper import run_Unsupervise_AD
from TSB_AD.utils.slidingWindows import find_length_rank

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Configuration (Edit This Cell)

This is the only cell you should normally edit.

After changing config values, rerun from the **Run Benchmark** cell onward.


In [17]:
CONFIG: dict[str, Any] = {
    "seed": 2026,
    "data_dir": Path("../datasets/TSB-AD-U"),
    "results_dir": Path("../results"),
    "summary_csv_name": "statistical_methods_summary.csv",
    "raw_csv_name": "statistical_methods_raw.csv",
    "selected_methods": [
        "Sub_PCA",
        "KShapeAD",
        "Series2Graph",
        "KMeansAD",
    ],
    "method_overrides": {
        "Sub_PCA": {"periodicity": 1},
        "KShapeAD": {"periodicity": 1},
        "Series2Graph": {"periodicity": 1},
        "KMeansAD": {"n_clusters": 20, "window_size": 20},
    },
    "file_selection": {
        "mode": "first_n",  # first_n | explicit_list | all
        "first_n": 10,
        "explicit_files": [],  # Example: ["001_NAB_id_1_Facility_tr_1007_1st_2014.csv"]
    },
    "evaluation_mode": "test",  # test | full | both
    "suppress_undefined_metric_warnings": True,
    "suppress_runtime_warnings": True,
    "verbose": True,
}

print("Configuration loaded. Edit values above if needed.")

Configuration loaded. Edit values above if needed.


## Method Registry and Metric Schema

Add new methods here, then include them in `CONFIG["selected_methods"]`.


In [18]:
METHOD_REGISTRY: dict[str, dict[str, Any]] = {
    "Sub_PCA": {
        "default_kwargs": {"periodicity": 1},
        "description": "Subsequence PCA-based detector",
    },
    "KShapeAD": {
        "default_kwargs": {"periodicity": 1},
        "description": "k-Shape/SAND based detector",
    },
    "Series2Graph": {
        "default_kwargs": {"periodicity": 1},
        "description": "Graph conversion anomaly detector",
    },
    "KMeansAD": {
        "default_kwargs": {"n_clusters": 20, "window_size": 20},
        "description": "Windowed k-means anomaly detector",
    },
}

METRIC_COLUMNS: list[str] = [
    "AUC-ROC",
    "AUC-PR",
    "Standard-F1",
    "PA-F1",
    "R-based-F1",
    "Affiliation-F1",
    "VUS-ROC",
    "VUS-PR",
]

METRIC_RENAME_MAP: dict[str, str] = {
    "Affiliation-F": "Affiliation-F1",
}

print(f"Registered methods: {list(METHOD_REGISTRY)}")

Registered methods: ['Sub_PCA', 'KShapeAD', 'Series2Graph', 'KMeansAD']


## Helper Functions

These functions are intentionally small and reusable so the run logic stays clean.


In [19]:
FIRST_TEST_PATTERN = re.compile(r"_1st_(\d+)\.csv$")


def set_global_seed(seed: int) -> None:
    np.random.seed(seed)
    random.seed(seed)


def resolve_data_dir(data_dir: Path | str) -> Path:
    resolved = Path(data_dir).expanduser().resolve()
    if not resolved.exists():
        raise FileNotFoundError(f"Data directory not found: {resolved}")
    return resolved


def list_candidate_files(data_dir: Path, file_selection_cfg: dict[str, Any]) -> list[Path]:
    all_files = sorted(data_dir.glob("*.csv"))
    if not all_files:
        raise ValueError(f"No CSV files found in {data_dir}")

    mode = str(file_selection_cfg.get("mode", "first_n")).lower()
    if mode == "all":
        return all_files

    if mode == "explicit_list":
        explicit_files = file_selection_cfg.get("explicit_files", [])
        selected: list[Path] = []
        for filename in explicit_files:
            candidate = data_dir / str(filename)
            if not candidate.exists():
                raise FileNotFoundError(f"Explicit file not found: {candidate}")
            selected.append(candidate)
        if not selected:
            raise ValueError("Mode 'explicit_list' selected but no explicit files configured")
        return selected

    if mode == "first_n":
        first_n = int(file_selection_cfg.get("first_n", 10))
        if first_n <= 0:
            raise ValueError("file_selection.first_n must be > 0")
        return all_files[:first_n]

    raise ValueError(f"Unsupported file selection mode: {mode}")


def parse_first_test_index(file_path: Path) -> int:
    match = FIRST_TEST_PATTERN.search(file_path.name)
    if match is None:
        return 0
    return int(match.group(1))


def load_univariate_file(file_path: Path) -> tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(file_path).dropna()
    if "Label" not in df.columns:
        raise ValueError(f"Missing 'Label' column in {file_path.name}")

    feature_cols = [col for col in df.columns if col != "Label"]
    if not feature_cols:
        raise ValueError(f"No feature column found in {file_path.name}")

    data = df[[feature_cols[0]]].astype(float).to_numpy()
    labels = df["Label"].astype(int).to_numpy()
    if len(data) != len(labels):
        raise ValueError(f"Data/label length mismatch in {file_path.name}")
    return data, labels


def resolve_eval_modes(mode: str) -> list[str]:
    mode = mode.lower()
    if mode == "both":
        return ["test", "full"]
    if mode in {"test", "full"}:
        return [mode]
    raise ValueError("evaluation_mode must be one of: test, full, both")


def compute_eval_slice(n_points: int, first_test_index: int, eval_mode: str) -> slice:
    if n_points <= 0:
        return slice(0, 0)
    if eval_mode == "full":
        return slice(0, n_points)
    start = max(0, min(first_test_index, n_points - 1))
    return slice(start, n_points)


def resolve_method_kwargs(method_name: str, method_overrides: dict[str, dict[str, Any]]) -> dict[str, Any]:
    default_kwargs = dict(METHOD_REGISTRY[method_name]["default_kwargs"])
    override_kwargs = dict(method_overrides.get(method_name, {}))
    default_kwargs.update(override_kwargs)
    return default_kwargs


def safe_sliding_window(data: np.ndarray) -> int:
    if len(data) < 3:
        return 1
    try:
        return max(1, int(find_length_rank(data, rank=1)))
    except Exception:
        return max(1, min(100, len(data) // 2))


def scale_scores(scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=float).reshape(-1, 1)
    if scores.size == 0:
        raise ValueError("Received empty score array")

    finite_mask = np.isfinite(scores).ravel()
    if not finite_mask.all():
        finite_vals = scores[finite_mask]
        replacement = float(np.median(finite_vals)) if finite_vals.size > 0 else 0.0
        scores = np.nan_to_num(scores, nan=replacement, posinf=replacement, neginf=replacement)

    if np.allclose(float(scores.min()), float(scores.max())):
        return np.zeros(scores.shape[0], dtype=float)

    return MinMaxScaler(feature_range=(0, 1)).fit_transform(scores).ravel()


def normalize_metric_dict(metrics: dict[str, Any]) -> dict[str, float]:
    normalized: dict[str, float] = {}
    for key, value in metrics.items():
        mapped_key = METRIC_RENAME_MAP.get(key, key)
        normalized[mapped_key] = float(value)

    for metric in METRIC_COLUMNS:
        normalized.setdefault(metric, np.nan)

    return normalized


## Run Benchmark

This cell uses the config and helpers above. You should not need to edit it for normal experiments.


In [20]:
set_global_seed(int(CONFIG["seed"]))

if bool(CONFIG.get("suppress_undefined_metric_warnings", True)):
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
if bool(CONFIG.get("suppress_runtime_warnings", True)):
    warnings.filterwarnings(
        "ignore",
        message="invalid value encountered in divide",
        category=RuntimeWarning,
    )

data_dir = resolve_data_dir(CONFIG["data_dir"])
selected_files = list_candidate_files(data_dir, CONFIG["file_selection"])
selected_methods = list(CONFIG["selected_methods"])

unknown_methods = [m for m in selected_methods if m not in METHOD_REGISTRY]
if unknown_methods:
    raise ValueError(f"Unknown methods in config: {unknown_methods}")

eval_modes = resolve_eval_modes(str(CONFIG["evaluation_mode"]))

print(f"Data directory: {data_dir}")
print(f"Selected files: {len(selected_files)}")
print(f"Selected methods (requested): {selected_methods}")
print(f"Evaluation modes: {eval_modes}")

run_rows: list[dict[str, Any]] = []
failure_rows: list[dict[str, Any]] = []
disabled_methods: set[str] = set()

run_start = time.time()

for file_idx, file_path in enumerate(selected_files, start=1):
    if CONFIG["verbose"]:
        print(f"[{file_idx}/{len(selected_files)}] {file_path.name}")

    try:
        data, labels = load_univariate_file(file_path)
    except Exception as exc:
        failure_rows.append(
            {
                "file": file_path.name,
                "method": "__load__",
                "evaluation_window": "n/a",
                "error": str(exc),
                "kwargs": {},
            }
        )
        if CONFIG["verbose"]:
            print(f"  load failed: {exc}")
        continue

    first_test_index = parse_first_test_index(file_path)

    for method_name in selected_methods:
        if method_name in disabled_methods:
            continue

        kwargs = resolve_method_kwargs(method_name, CONFIG["method_overrides"])
        method_start = time.perf_counter()

        try:
            raw_scores = run_Unsupervise_AD(method_name, data, **kwargs)
            if isinstance(raw_scores, str):
                if "is not defined" in raw_scores:
                    disabled_methods.add(method_name)
                    raise RuntimeError(
                        f"{raw_scores} Method will be skipped for remaining files."
                    )
                raise RuntimeError(raw_scores)

            raw_scores = np.asarray(raw_scores).ravel()
            if raw_scores.size == 0:
                raise ValueError("Model returned empty scores")

            aligned_len = min(len(raw_scores), len(labels))
            if aligned_len < 2:
                raise ValueError(
                    f"Insufficient overlap between scores ({len(raw_scores)}) and labels ({len(labels)})"
                )

            if aligned_len != len(labels) and CONFIG["verbose"]:
                print(
                    f"  {method_name}: score length {len(raw_scores)} != label length {len(labels)}; trimming to {aligned_len}"
                )

            data_aligned = data[:aligned_len]
            labels_aligned = labels[:aligned_len]
            scaled_scores = scale_scores(raw_scores[:aligned_len])

        except Exception as exc:
            failure_rows.append(
                {
                    "file": file_path.name,
                    "method": method_name,
                    "evaluation_window": "n/a",
                    "error": str(exc),
                    "kwargs": kwargs,
                }
            )
            if CONFIG["verbose"]:
                print(f"  {method_name} failed before evaluation: {exc}")
            continue

        for eval_mode in eval_modes:
            try:
                eval_slice = compute_eval_slice(len(labels_aligned), first_test_index, eval_mode)
                score_slice = scaled_scores[eval_slice]
                label_slice = labels_aligned[eval_slice]
                data_slice = data_aligned[eval_slice]

                if len(score_slice) < 2:
                    raise ValueError(f"Not enough points after {eval_mode} slicing")
                if np.unique(label_slice).size < 2:
                    raise ValueError(f"Only one class present after {eval_mode} slicing")

                sliding_window = safe_sliding_window(data_slice)
                metrics = get_metrics(score_slice, label_slice, slidingWindow=sliding_window)
                normalized_metrics = normalize_metric_dict(metrics)

                elapsed = float(time.perf_counter() - method_start)
                row = {
                    "file": file_path.name,
                    "method": method_name,
                    "evaluation_window": eval_mode,
                    "n_points": int(len(label_slice)),
                    "n_anomalies": int(label_slice.sum()),
                    "first_test_index": int(first_test_index),
                    "runtime_sec": elapsed,
                    "status": "ok",
                }
                row.update({metric: normalized_metrics[metric] for metric in METRIC_COLUMNS})
                run_rows.append(row)

            except Exception as exc:
                failure_rows.append(
                    {
                        "file": file_path.name,
                        "method": method_name,
                        "evaluation_window": eval_mode,
                        "error": str(exc),
                        "kwargs": kwargs,
                    }
                )
                if CONFIG["verbose"]:
                    print(f"  {method_name} ({eval_mode}) failed: {exc}")

raw_results_df = pd.DataFrame(run_rows)
failures_df = pd.DataFrame(failure_rows)

total_runtime = time.time() - run_start
print(f"\nFinished in {total_runtime:.2f}s")
print(f"Successful runs: {len(raw_results_df)}")
print(f"Failed runs: {len(failures_df)}")
if disabled_methods:
    print(f"Disabled methods in this run: {sorted(disabled_methods)}")

if not raw_results_df.empty:
    display(raw_results_df.head())
if not failures_df.empty:
    display(failures_df.head())

Data directory: /home/lukas/Code/Master/tsb-ad-test/datasets/TSB-AD-U
Selected files: 10
Selected methods (requested): ['Sub_PCA', 'KShapeAD', 'Series2Graph', 'KMeansAD']
Evaluation modes: ['test']
[1/10] 001_NAB_id_1_Facility_tr_1007_1st_2014.csv
Model function 'run_Series2Graph' is not defined.
  Series2Graph failed before evaluation: Model function 'run_Series2Graph' is not defined. Method will be skipped for remaining files.
Required padding_length=0
Reversing window-based scores to point-based scores:
Before reverse-windowing: scores.shape=(4012,)
After reverse-windowing: scores.shape=(4031,)
[2/10] 002_NAB_id_2_WebService_tr_1500_1st_4106.csv
Required padding_length=0
Reversing window-based scores to point-based scores:
Before reverse-windowing: scores.shape=(5981,)
After reverse-windowing: scores.shape=(6000,)
[3/10] 003_NAB_id_3_WebService_tr_1362_1st_1462.csv
Required padding_length=0
Reversing window-based scores to point-based scores:
Before reverse-windowing: scores.shape=(

Traceback (most recent call last):
  File "/home/lukas/Code/Master/tsb-ad-test/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_121130/2360758848.py", line 118, in <module>
    metrics = get_metrics(score_slice, label_slice, slidingWindow=sliding_window)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lukas/Code/Master/tsb-ad-test/.venv/lib/python3.11/site-packages/TSB_AD/evaluation/metrics.py", line 26, in get_metrics
    RF1 = grader.metric_RF1(labels, score, preds=pred)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lukas/Code/Master/tsb-ad-test/.venv/lib/python3.11/site-packages/TSB_AD/evaluation/basic_metrics.py", line 267, in metric_RF1
    Rprecision = self.range_recall_new(preds, label, 0)[0]
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/lukas/Code/Master/tsb-ad-test

## Aggregate Results

This creates a method-level summary table from successful runs.


In [ ]:
if raw_results_df.empty:
    summary_df = pd.DataFrame()
    print("No successful runs found. Check failures_df for details.")
else:
    metric_summary = (
        raw_results_df
        .groupby(["method", "evaluation_window"], as_index=False)[METRIC_COLUMNS]
        .mean()
    )

    count_summary = (
        raw_results_df
        .groupby(["method", "evaluation_window"], as_index=False)
        .agg(
            files_succeeded=("file", "nunique"),
            runs=("file", "size"),
            avg_runtime_sec=("runtime_sec", "mean"),
            total_points=("n_points", "sum"),
            total_anomalies=("n_anomalies", "sum"),
        )
    )

    summary_df = (
        count_summary
        .merge(metric_summary, on=["method", "evaluation_window"], how="left")
        .sort_values(["evaluation_window", "method"])
        .reset_index(drop=True)
    )

    display(summary_df)

if not failures_df.empty:
    print("\nFailures (showing up to 20 rows):")
    display(failures_df.head(20))

,method,evaluation_window,files_succeeded,runs,avg_runtime_sec,total_points,total_anomalies,AUC-ROC,AUC-PR,Standard-F1,PA-F1,R-based-F1,Affiliation-F1,VUS-ROC,VUS-PR
0,KMeansAD,test,10,10,1.061470,30270,5187,0.500155,0.320212,0.428941,0.909049,0.521858,0.804189,0.519451,0.338336
1,KShapeAD,test,9,9,2.307041,29541,5039,0.427740,0.358323,0.486830,0.801713,0.673821,0.791324,0.440197,0.371084
2,Sub_PCA,test,10,10,0.758212,30270,5187,0.707203,0.591360,0.605163,0.938731,0.672966,0.924531,0.707702,0.590094



Failures (showing up to 20 rows):


,file,method,evaluation_window,error,kwargs
0,001_NAB_id_1_Facility_tr_1007_1st_2014.csv,Series2Graph,n/a,Model function 'run_Series2Graph' is not defin...,{'periodicity': 1}
1,010_NAB_id_10_WebService_tr_500_1st_271.csv,KShapeAD,n/a,Model function 'run_KShapeAD' is not defined. ...,{'periodicity': 1}


## Export Results

This cell writes summary and raw run tables to CSV files under `CONFIG["results_dir"]`.


In [ ]:
results_dir = Path(CONFIG["results_dir"]).expanduser().resolve()
results_dir.mkdir(parents=True, exist_ok=True)

summary_path = results_dir / str(CONFIG["summary_csv_name"])
raw_path = results_dir / str(CONFIG["raw_csv_name"])

if "summary_df" in globals() and not summary_df.empty:
    summary_df.to_csv(summary_path, index=False)
    print(f"Summary CSV written to: {summary_path}")
else:
    print("Summary CSV not written because summary_df is empty.")

if "raw_results_df" in globals() and not raw_results_df.empty:
    raw_results_df.to_csv(raw_path, index=False)
    print(f"Raw results CSV written to: {raw_path}")
else:
    print("Raw CSV not written because raw_results_df is empty.")

Summary CSV written to: /home/lukas/Code/Master/tsb-ad-test/results/statistical_methods_summary.csv
Raw results CSV written to: /home/lukas/Code/Master/tsb-ad-test/results/statistical_methods_raw.csv


## Extension Notes

To add a new method later:
1. Add it in `METHOD_REGISTRY` with default kwargs.
2. Add the method name to `CONFIG["selected_methods"]`.
3. Optionally add per-method overrides in `CONFIG["method_overrides"]`.

No further notebook rewrite should be necessary for normal parameter tuning.